In [ ]:
import numpy as np
import proplot as pplt
import os
import random

fitting_dir = 'your_directory/loads/com_fitting'
load_dir = 'your_directory/loads/new'
weather_dir = 'your_directory/loads/weather'

fitting_files = sorted([f for f in os.listdir(fitting_dir) if f.endswith('.npz')])
chosen_file = 'fitting_36001.npz'
county_code = chosen_file.replace('fitting_', '').replace('.npz', '')
print(f"county: {county_code}")

fitting_data = np.load(os.path.join(fitting_dir, chosen_file))
heating_stats = fitting_data['heating']
cooling_stats = fitting_data['cooling']
heating_thresholds = fitting_data['heating_threshold']
cooling_thresholds = fitting_data['cooling_threshold']

load_data = np.load(os.path.join(load_dir, f'load_{county_code}.npz'))
weather_data = np.load(os.path.join(weather_dir, f'weather_{county_code}.npz'))['data']
temperature = weather_data[0:0 + 8760, 5]

heating_load = load_data['com_heating'] / 1000
cooling_load = load_data['com_cooling'] / 1000

heat_by_hour = heating_load.reshape((365, 24))
cool_by_hour = cooling_load.reshape((365, 24))
temp_by_hour = temperature.reshape((365, 24))

fig, axs = pplt.subplots(ncols=4, nrows=6, figsize=(8, 10))
fig.patch.set_facecolor('white')
axs.format(abc='a.')
for hour in range(24):
    ax = axs[hour // 4, hour % 4]  # Calculate row and column index for proplot grid

    x = temp_by_hour[:, hour]
    y_heat = heat_by_hour[:, hour]
    y_cool = cool_by_hour[:, hour]

    # Scatter plots for Heating and Cooling
    if hour==0:
        ax.scatter(x, y_heat, color=[186/255.0,230/255.0,250/255.0], alpha=0.5, label='Heating', zorder=2)
        ax.scatter(x, y_cool, color=[251/255.0,202/255.0,197/255.0], alpha=0.5, label='Cooling', zorder=2)
    else:
        ax.scatter(x, y_heat, color=[186/255.0,230/255.0,250/255.0], alpha=0.5, zorder=2)
        ax.scatter(x, y_cool, color=[251/255.0,202/255.0,197/255.0], alpha=0.5, zorder=2)

    # Heating Fit
    ph = heating_stats[hour, :3]
    threshold_h = heating_thresholds[hour]

    if not np.any(np.isnan(ph)) and not np.isnan(threshold_h):
        xh_range = np.linspace(x.min(), threshold_h, 100)
        yh_pred = ph[0]*xh_range**2 + ph[1]*xh_range + ph[2]
        if hour == 0:
            ax.plot(xh_range, yh_pred, color=[76/255.0,152/255.0,206/255.0], lw=2, label=f'Heating Fit', zorder=3)
        else:
            ax.plot(xh_range, yh_pred, color=[76/255.0,152/255.0,206/255.0], lw=2, zorder=3)

    # Cooling Fit
    pc = cooling_stats[hour, :3]
    threshold_c = cooling_thresholds[hour]

    if not np.any(np.isnan(pc)) and not np.isnan(threshold_c):
        xc_range = np.linspace(threshold_c, x.max(), 100)
        yc_pred = pc[0]*xc_range**2 + pc[1]*xc_range + pc[2]
        if hour ==0:
            ax.plot(xc_range, yc_pred, color=[235/255.0,157/255.0,158/255.0], lw=2, label=f'Cooling Fit', zorder=3)
        else:
            ax.plot(xc_range, yc_pred, color=[235/255.0,157/255.0,158/255.0], lw=2, zorder=3)

    ax.set_title(f'Hour {hour:02d}:00')
    ax.set_xlabel('Temperature (×10 °C)')
    ax.set_ylabel('Load (MW)')
    ax.grid(True)

fig.legend(loc='top', ncol=4, fontsize=12, frameon=False)
fig.tight_layout(rect=[0, 0, 1, 0.97])
pplt.show()
